# 01 TLM preparation

Builds `TLM_total` by combining three layers of swissTLM3D:

- land cover from TLM Bodenbedeckung,
- settlement areas from TLM Siedlungsname,
- orchards and vineyards from TLM Nutzungsareal.

The result feeds notebook 02, which combines it with the yearly cantonal
agricultural land use data.

Run this once per swissTLM3D release. The TLM is updated irregularly, roughly
yearly, so this is not part of the annual routine.

**What changed in this version.** The processing logic now lives in the `tlm_ln`
package next to this notebook, and paths live in `config.toml`. The notebook is a
driver: it runs the steps, shows the maps, and reports the checks. That way the
logic can be tested, and editing a path no longer means editing code.

## 1. Setup

Requires `geopandas`, `pandas`, `shapely`, `matplotlib` and `pyogrio`.

```bash
pip install geopandas pandas shapely matplotlib pyogrio
```

In [ ]:
import matplotlib.pyplot as plt

import tlm_ln
from tlm_ln import config, pipeline

tlm_ln.setup_logging()

In [ ]:
def plot_map(gdf, title, column=None, max_features=5000, figsize=(9, 9)):
    """Quick visual check of a layer.

    Large layers are sampled for the plot only; the data itself is untouched.
    Kept in the notebook rather than in the package because plotting is a
    property of how you are looking at the data, not of the workflow.
    """
    if gdf.empty:
        print(f"{title}: layer is empty, nothing to plot")
        return
    subset = gdf.sample(max_features, random_state=42) if len(gdf) > max_features else gdf
    if len(subset) < len(gdf):
        print(f"plotting {max_features:,} of {len(gdf):,} features")
    fig, ax = plt.subplots(figsize=figsize)
    if column and column in subset.columns:
        subset.plot(ax=ax, column=column, legend=True, linewidth=0.2, markersize=2)
    else:
        subset.plot(ax=ax, linewidth=0.2, markersize=2)
    ax.set_title(title)
    ax.set_axis_off()
    plt.show()

## 2. Configuration

Paths live in `config.toml` beside this notebook. Open it to point at your copy
of swissTLM3D and the lookup table. Nothing below needs editing.

In [ ]:
cfg, raw = config.load("config.toml")

print(f"TLM GeoPackage : {cfg.tlm_gpkg}")
print(f"Lookup table   : {cfg.tlm_lookup_table}")
print(f"Output         : {cfg.output_gpkg}")
print(f"CRS            : {cfg.crs}")

## 3. Run

One call does the whole workflow: classify each source, merge land cover with
orchards and vineyards, erase settlements by that merge, and merge the erased
settlements back in.

The slow steps are reading Bodenbedeckung and the erase, each a few minutes on
national data. Watch the log for progress. Every step reports its feature count
and area, so an unexpected result shows up while it happens rather than at the
end.

Set `write=False` to try it without touching the output GeoPackage.

In [ ]:
result = pipeline.prepare_tlm(cfg, write=True)

layers = result["layers"]

## 4. Checks

Read the area table first. Two things to look for:

- `TLM_total` should be roughly `Siedl_erase` plus `TLM_ObstReben`. A large
  discrepancy means geometries were lost somewhere.
- `Siedl_erase` should be noticeably smaller than `TLM_Siedl` but not empty. If it
  is empty, TLM Bodenbedeckung has started covering settlements and this step
  needs rethinking.

In [ ]:
result["areas"]

In [ ]:
# Features carrying no classification. These are invisible to any downstream
# selection, so they are effectively missing even though they are in the file.
result["completeness"]

In [ ]:
# Area per class. The quickest way to spot a class that has vanished or doubled
# between TLM releases.
result["by_class"]

## 5. Maps

Visual confirmation that each step did what the diagram says.

In [ ]:
plot_map(layers["TLM_Reben"], "TLM_Reben: orchards and vineyards", column="group_code")

In [ ]:
plot_map(layers["TLM_Siedl"], "TLM_Siedl: settlement polygons before erase", column="class_code")
plot_map(layers["Siedl_erase"], "Siedl_erase: settlement polygons after erase", column="class_code")

In [ ]:
# Bodenbedeckung is large. Plotting it is slow even sampled, so it is left
# commented out; uncomment when you want to look.
# plot_map(layers["TLM_BB"], "TLM_BB: land cover classes", column="class_code")
plot_map(layers["TLM_total"], "TLM_total", column="class_code")

## 6. What was written

Alongside the layers, the run writes `<output>.manifest.json` recording the input
files with their checksums, the package versions, and the settings used. That is
what lets someone tie this GeoPackage back to the exact inputs that produced it,
months later, without relying on memory.

In [ ]:
import json
from pathlib import Path

manifest_path = cfg.output_gpkg.with_suffix(".manifest.json")
if manifest_path.exists():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2)[:2000])